In [1]:
# Initiate session.

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 08:14:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/17 08:14:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Read yellow taxi file.

df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [5]:
# Normalize column names and add a column for service type.

from pyspark.sql import functions as F

df_trips_data = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('service_type', F.lit('yellow'))
df_trips_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- service_type: string (nullable = false)



In [11]:
# Crystalise the table.

df_trips_data.groupBy('service_type', 'VendorID').count().show()

+------------+--------+-------+
|service_type|VendorID|  count|
+------------+--------+-------+
|      yellow|       1| 821333|
|      yellow|       2|3295835|
|      yellow|       7|  60043|
|      yellow|       6|   4233|
+------------+--------+-------+



In [12]:
# Register temp table to enable Spark SQL.

df_trips_data.createOrReplaceTempView('trips_data')

In [13]:
# Test SQL - output should match the output from 2 prompts above.

spark.sql("""
select
    service_type,
    VendorID,
    count(1)
from trips_data
group by
    service_type,
    VendorID
limit 10;
""").show()

[Stage 11:=============================>                            (2 + 2) / 4]

+------------+--------+--------+
|service_type|VendorID|count(1)|
+------------+--------+--------+
|      yellow|       1|  821333|
|      yellow|       2| 3295835|
|      yellow|       7|   60043|
|      yellow|       6|    4233|
+------------+--------+--------+



In [19]:
# Define Spark query.

df_result = spark.sql("""
select
    -- Grouping dimensions
    PULocationID as pickup_zone,
    date_trunc('month', pickup_datetime) as revenue_month,
    service_type,

    -- Revenue breakdown (summed by zone, month, and service type)
    sum(fare_amount) as revenue_monthly_fare,
    sum(extra) as revenue_monthly_extra,
    sum(mta_tax) as revenue_monthly_mta_tax,
    sum(tip_amount) as revenue_monthly_tip_amount,
    sum(tolls_amount) as revenue_monthly_tolls_amount,
    sum(improvement_surcharge) as revenue_monthly_improvement_surcharge,
    sum(total_amount) as revenue_monthly_total_amount,
    sum(congestion_surcharge) as revenue_monthly_congestion_surcharge,
    sum(Airport_fee) as revenue_monthly_Airport_fee,
    sum(cbd_congestion_fee) as revenue_monthly_cbd_congestion_fee,

    -- Additional metrics for operational analysis
    avg(passenger_count) as avg_monthly_passenger_count,
    avg(trip_distance) as avg_monthly_trip_distance

from
    trips_data
group by
    pickup_zone, revenue_month, service_type;
""")

In [20]:
df_result.show()

[Stage 17:=============================>                            (2 + 2) / 4]

+-----------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+----------------------------------+---------------------------+-------------------------+
|pickup_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|revenue_monthly_Airport_fee|revenue_monthly_cbd_congestion_fee|avg_monthly_passenger_count|avg_monthly_trip_distance|
+-----------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+-----------

In [23]:
# Save down the results of the query.

df_result.coalesce(1).write.parquet('data/report/revenue', mode='overwrite')